# 01 — Replay store_db history into Kafka

`store_db` is a *static*, already fully-generated year (2025-01-01 -> 2025-12-31),
not a live system. This notebook reads every event-bearing row out of Postgres in
true chronological order and republishes it to Kafka (a message broker — software
that lets independent programs publish and subscribe to streams of events) as if
it were happening live.

Run the cells below in order. `build_timeline()` only needs to run once per kernel
session — after that, keep re-running the `publish_next(...)` cell with different
arguments; each call resumes exactly where the previous one left off.

## Setup

Make the `producers` package importable (this notebook lives in `notebooks/`,
one level below the project root).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from producers.replay import build_timeline, publish_next

## Build the timeline (run once)

Queries every source table in `store_db`, merges everything into one
chronologically sorted list of events, and returns an iterator over it. Keep
`events` around for the rest of the session — every `publish_next()` call below
pulls from it and advances it.

In [2]:
events = build_timeline()

Timeline built: 4991 events total.


## Smoke test — publish a small batch fast

Always verify with a small `limit` first. Never pass `limit=None` while just
checking things look right — that drains the *entire* timeline at once.

In [3]:
publish_next(events, limit=5)

Published 5 messages this call:
  inventory.restocked: 5


{'inventory.restocked': 5}

Check `http://localhost:8089` (kafka-ui) now and confirm those 20 messages landed
across the expected topics.

## Paced replay — watch it drip out slowly

Publish a batch of messages one at a time, a few seconds apart. Run this cell
while `02_consume.ipynb`'s `consume_batch()` cell is also running (in another
notebook/kernel), and watch bronze Parquet (a columnar file format good for
analytics) files appear as the consumer picks things up.

In [4]:
publish_next(events, limit=30, interval_seconds=5, batch_size=3)

Published 30 messages this call:
  inventory.restocked: 7
  price.changed: 23


{'inventory.restocked': 7, 'price.changed': 23}

## Drain the rest

Once you're done exploring pacing, publish everything still left in the
timeline. Safe to re-run — once `events` is exhausted, further calls just
report 0 published.

In [ ]:
publish_next(events)